### Import Libraries and Fetch Webpage Data


In [1]:
# Import necessary external libraries for scraping and data handling
from bs4 import BeautifulSoup  # For parsing raw HTML text structures
import pandas as pd            # For creating DataFrames and exporting to CSV
import requests                # For making HTTP requests to download web content

In [2]:
# Target Wikipedia URL containing the company financial tables
url = "https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue"

In [3]:
# Add browser headers to mimic a normal user visit and prevent HTTP 403 blocks
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [4]:
# Download the live webpage content from the target URL
page = requests.get(url, headers=headers)

In [5]:
# Initialize BeautifulSoup to parse the raw HTML document tree
soup = BeautifulSoup(page.text, "html.parser")

In [6]:
# Locate the very first data table element found on the webpage
table = soup.find_all('table')[0]

In [7]:
# Print only the first 500 characters of the formatted HTML table as a clean preview
print(table.prettify()[:500])

<table class="wikitable sortable">
 <caption>
 </caption>
 <tbody>
  <tr>
   <th>
    Rank
   </th>
   <th>
    Name
   </th>
   <th>
    Industry
   </th>
   <th>
    Revenue
    <br/>
    (USD millions)
   </th>
   <th>
    Revenue growth
   </th>
   <th>
    Employees
   </th>
   <th>
    Headquarters
   </th>
  </tr>
  <tr>
   <td>
    1
   </td>
   <td>
    <a href="/wiki/Walmart" title="Walmart">
     Walmart
    </a>
   </td>
   <td>
    Retail
   </td>
   <td style="text-align:center;">



### Extract Table Column Headers

In [8]:
# Find all table header tags (<th>) inside the isolated target table
header = table.find_all('th')

In [9]:
# Loop through each header element, extract clean text, and store in a list
title = []
for head in header:
    name = head.get_text(strip=True)  # strip=True drops trailing spaces or newlines
    title.append(name)

In [10]:
# Display the final list of discovered table column headers
title

['Rank',
 'Name',
 'Industry',
 'Revenue(USD millions)',
 'Revenue growth',
 'Employees',
 'Headquarters']

### Initialize the Empty DataFrame

In [11]:
# Create an empty Pandas DataFrame mapped to the extracted column names
df = pd.DataFrame(columns=title)

# Display the empty DataFrame structure to verify layout positioning
df

,Rank,Name,Industry,Revenue(USD millions),Revenue growth,Employees,Headquarters


### Parse Row Content Data

In [12]:
# Locate all structural table rows (<tr>) from the targeted HTML table
rows = table.find_all("tr") 

In [13]:
# Loop through every row, skipping the first row index to bypass headers
table_data = []
for row in rows[1:]:
    cells = row.find_all("td") # Find individual table data cells (<td>) in current row
    
    # Process and append cell data to our collection if the row is not empty
    if cells:
        row_data = [cell.get_text(strip=True) for cell in cells]
        table_data.append(row_data)

In [14]:
# Preview the first two parsed data arrays to verify successful data cleaning
table_data[:2]

[['1',
  'Walmart',
  'Retail',
  '680,985',
  '5.1%',
  '2,100,000',
  'Bentonville, Arkansas'],
 ['2',
  'Amazon',
  'Retail and cloud computing',
  '637,959',
  '11.0%',
  '1,556,000',
  'Seattle, Washington']]

In [15]:
df = pd.DataFrame(table_data, columns=title)

In [16]:
df.head()

,Rank,Name,Industry,Revenue(USD millions),Revenue growth,Employees,Headquarters
0,1,Walmart,Retail,"680,985",5.1%,"2,100,000","Bentonville, Arkansas"
1,2,Amazon,Retail and cloud computing,"637,959",11.0%,"1,556,000","Seattle, Washington"
2,3,UnitedHealth Group,Healthcare,"400,278",7.7%,"400,000","Minnetonka, Minnesota"
3,4,Apple,Technology,"391,035",2.0%,"164,000","Cupertino, California"
4,5,CVS Health,Healthcare,"372,809",4.2%,"259,500","Woonsocket, Rhode Island"


In [17]:
df.tail()

,Rank,Name,Industry,Revenue(USD millions),Revenue growth,Employees,Headquarters
95,96,General Dynamics,Aerospace and defense,"47,716",12.9%,"117,000","Reston, Virginia"
96,97,Coca-Cola,Beverage,"47,061",2.9%,"69,700","Atlanta, Georgia"
97,98,TIAA,Financials,"46,946",2.6%,"15,623","New York City, New York"
98,99,Travelers,Insurance,"46,423",12.2%,"34,000","New York City, New York"
99,100,Eli Lilly,Pharmaceutical,"45,043",32.0%,"47,000","Indianapolis, Indiana"


In [18]:
df.to_csv("2_Company_Revenue_Scraper.csv", index=False)